# NeMo Relay, Langfuse OTLP, and Decision Model Evaluation Tutorial

This notebook demonstrates the unified observability and evaluation architecture in **GenAI Toolkit**:
1. **The Telemetry Core**: Using NVIDIA NeMo Relay to capture hierarchical Agent Trajectory Observability Format (ATOF) records.
2. **OTLP Export to Langfuse**: Streaming OpenTelemetry traces directly to Langfuse using NeMo Relay's native Rust/C++ exporter.
3. **Trajectory Inspection**: Programmatically querying recorded runs, LLM calls, and tool executions with `TrajectoryStore`.
4. **Deterministic Tool Matching**: Verifying agent tool call sequences without external library overhead.
5. **System One Decision Models**: Evaluating agent outcomes with typed binary judgments (`Noul`), categorical choices (`Choice`), and ordinal rubrics (`Score`).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
sys.path.append(str(Path.cwd().parent.parent))

## 1. NeMo Relay Observability & OTLP Setup

NeMo Relay provides native scopes (`ScopeType.Agent`, `ScopeType.Llm`, `ScopeType.Tool`, `ScopeType.Evaluator`, `ScopeType.Function`).
When initialized, it writes ATOF JSONL events to a local per-session store and streams OpenTelemetry (OTLP) traces to Langfuse (`http://localhost:3000/api/public/otel/v1/traces`).

In [ ]:
from genai_tk.extra.monitoring.nemo_relay_setup import (
    emit_test_probe,
    get_relay_otlp_endpoint,
    is_nemo_relay_available,
    is_relay_otlp_active,
    setup_nemo_relay,
)

print("NeMo Relay available:", is_nemo_relay_available())

demo_store_dir = Path("data/trajectories_demo")
setup_nemo_relay(store_dir=demo_store_dir, enable_otlp=False)

print("OTLP forwarder active:", is_relay_otlp_active())
print("OTLP target endpoint:", get_relay_otlp_endpoint())

# Emit a test probe through NeMo Relay
emit_test_probe("notebook-setup-probe")
print("Emitted probe successfully!")

## 2. Defining and Running a Scoped Deep Code Agent

Let's define a tool (`python_interpreter`) and run an agent. The agent call hierarchy is automatically mapped to NeMo Relay scopes using `NemoRelayDeepAgentsCallbackHandler`.

In [ ]:
from langchain_core.messages import AIMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver

from genai_tk.agents.langchain.config import AgentProfileConfig
from genai_tk.agents.langchain.factory import _create_deep_agent
from genai_tk.extra.monitoring.nemo_relay_setup import (
    flush_nemo_relay_async,
    get_relay_callback_handler,
)
from tests.integration_tests.test_deep_code_agent_trajectory_e2e import ScriptedCodeChatModel


@tool
def python_interpreter(code: str) -> str:
    """Execute Python code and return the output."""
    local_scope = {}
    try:
        exec(code, {}, local_scope)
        return str(local_scope.get("result", "executed"))
    except Exception as e:
        return f"Error: {e}"


# Simulate CodeAgent: tool call to python_interpreter, then final synthesis
scripted_model = ScriptedCodeChatModel(
    responses=[
        AIMessage(
            content="",
            tool_calls=[
                {
                    "name": "python_interpreter",
                    "args": {"code": "result = sum(range(1, 11))"},
                    "id": "call_sum_10",
                    "type": "tool_call",
                }
            ],
        ),
        AIMessage(content="The sum of integers from 1 to 10 is 55."),
    ]
)

profile = AgentProfileConfig(
    name="demo-code-agent",
    type="deep",
    llm="fake",
    tools=[],
    mcp_servers=[],
    skill_directories=[],
    enable_planning=False,
    enable_file_system=False,
)

agent = await _create_deep_agent(
    scripted_model,
    [python_interpreter],
    MemorySaver(),
    profile,
    middlewares=[],
    backend=None,
)

handler = get_relay_callback_handler()

user_query = "Calculate the sum of numbers from 1 to 10 using Python."
res = await agent.ainvoke(
    {"messages": user_query},
    config={"configurable": {"thread_id": "demo-thread-1"}, "callbacks": [handler]},
)

# Flush queued events to disk
await flush_nemo_relay_async()
print("Agent finished execution!")
print("Final response:", res["messages"][-1].content)

## 3. Programmatic Trajectory Inspection

The `TrajectoryStore` provides a typed read layer over the ATOF event stream. We can inspect the run hierarchy, turns, token metrics, and tool execution without re-executing the model.

In [ ]:
from genai_tk.extra.monitoring.trajectory_store import TrajectoryStore

store = TrajectoryStore(root=demo_store_dir)
runs = store.list_runs()
print(f"Total recorded runs: {len(runs)}")

run = runs[0]
print(f"Run ID: {run.run_id}")
print(f"Profile: {run.profile}")
print(f"Tools called: {run.tools}")
print(f"Token usage: {run.total_prompt_tokens} in / {run.total_completion_tokens} out")

traj = store.get(run.run_id)
print("\nTurn details:")
for turn in traj.turns:
    print(f"Turn #{turn.index}:")
    if turn.llm_call:
        print(f"  LLM: {turn.llm_call.model}")
    for tc in turn.tool_calls:
        print(f"  Tool: {tc.name}({tc.args}) -> {tc.result}")

## 4. Deterministic Trajectory Matching

To test agent adherence to expected tools without external dependencies (such as `agentevals`), use `match_trajectory_tools` or `traj.match_tools`.

In [ ]:
# Check superset: agent must have called python_interpreter
assert traj.match_tools(["python_interpreter"], mode="superset")
print("✓ Superset match passed!")

# Check strict sequence
assert traj.match_tools(["python_interpreter"], mode="strict")
print("✓ Strict sequence match passed!")

# Check that non-called tools fail
assert not traj.match_tools(["web_search"], mode="superset")
print("✓ Negative test passed!")

## 5. System One Decision Model Evaluations

Instead of relying on prompt-based LLM judges (like `openevals`) which output freeform prose requiring regex parsing, we use **System One Decision Models** (`Noul`, `Choice`, `Score`). These return calibrated probabilities and typed structured responses in <300ms.

In [ ]:
from genai_tk.core.decision_models.evaluators import (
    evaluate_conciseness,
    evaluate_correctness,
    evaluate_tool_selection,
)
from genai_tk.core.decision_models.fake import FakeDecisionModel

# For offline demonstration, we use FakeDecisionModel; in production use get_decision_model("clef_flash@openrouter")
decision_judge = FakeDecisionModel(fixed_noul=0.99, fixed_score=2.0, fixed_choice="optimal")

# 1. Correctness evaluation (NoulAnswer with probability)
c_eval = evaluate_correctness(
    decision_judge,
    question=user_query,
    gold_answer="55",
    agent_answer=res["messages"][-1].content,
)
print("Correctness Verdict:")
print(f"  Type: {c_eval.type}")
print(f"  Confidence / Probability: {c_eval.noul:.2%}")

# 2. Conciseness evaluation (ScoreAnswer on 0..2 scale)
s_eval = evaluate_conciseness(
    decision_judge,
    question=user_query,
    agent_answer=res["messages"][-1].content,
)
print("\nConciseness Verdict:")
print(f"  Score: {s_eval.score} ({s_eval.legend.get(int(s_eval.score))})")

# 3. Tool selection evaluation (ChoiceAnswer)
t_eval = evaluate_tool_selection(
    decision_judge,
    task=user_query,
    available_tools=["python_interpreter", "web_search", "calculator"],
    selected_tools=[tc.name for tc in traj.tool_calls],
)
print("\nTool Selection Quality:")
print(f"  Selection: {t_eval.choice}")

## Summary

- **NeMo Relay** is the unified telemetry backbone: in-process, zero-loss, hierarchical scopes for Agents, LLMs, Tools, and Evaluators.
- **Langfuse** receives clean OTLP spans directly from Relay's native exporter without runtime monkey-patching.
- **TrajectoryStore** gives full programmatic access to recorded trajectories for offline analysis and debugging.
- **Decision Models** replace brittle prompt-based judges with fast, typed, calibrated evaluators that feed directly into self-improving agent loops.